# Regressions

## Linear regressions

## Part 1: Income and education

**Import the Duncan/carData dataset**

In [ ]:
# Import statsmodels and load the Duncan dataset
import statsmodels.api as sm
dataset = sm.datasets.get_rdataset("Duncan", "carData", cache=True)
df = dataset.data
df.head(3)  
#df.info()  

In [ ]:
df.shape


In [ ]:
# Summary statistics
df.describe()

**Estimate by hand the model $\text{income} = \alpha + \beta \times \text{education}$. Plot.**

## OLS Reminder

Linear model: $y_i = \alpha + \beta x_i + \varepsilon_i$

**Formulas:**  
Slope: $\beta = \frac{Cov(y,x)}{Var(x)}$  
Intercept: $\alpha = \bar y - \beta \bar x$  
Prediction: $\hat y_i = \alpha + \beta x_i$

Manual calculation for demonstration (libraries handle this automatically):

In [ ]:
# Covariance matrix for OLS formula
Σ = df[['income','education']].cov()
Σ

In [ ]:
# Sample means
μ = df[['income','education']].mean()
μ

In [ ]:
# Slope coefficient β = Cov(y,x) / Var(x)
β = Σ.loc['income','education'] / Σ.loc['education','education']
β

In [ ]:
# Intercept α = mean(y) - β * mean(x)
α = μ['income'] - β*μ['education']
α

In [ ]:
# Predicted values: ŷ = α + β * education
prediction = α + β*df['education']
prediction

In [ ]:
# Plot regression
from matplotlib import pyplot as plt
plt.plot(df['education'], df['income'], 'o')  # Plot observed data points
plt.plot(df['education'], prediction, '-')  # Plot fitted regression line
plt.xlabel('Education')  # Label x-axis
plt.ylabel('Income')   # Label y-axis

**Compute total, explained, unexplained variance. Compute R² statistics**

#### Variance Decomposition & $R^2$

$y$ = actual, $\hat y$ = predicted, $e_i = y_i - \hat y_i$ = residual

**Decomposition:**  
$TSS = \sum_i (y_i - \bar y)^2$, $ESS = \sum_i (\hat y_i - \bar y)^2$, $RSS = \sum_i e_i^2$  
$TSS = ESS + RSS$

**Goodness of Fit:** $R^2 = \frac{ESS}{TSS} = 1 - \frac{RSS}{TSS}$

Manual calculation to verify formulas:

In [ ]:
# Store fitted values and residuals
df['prediction'] = α + β*df['education']
df['residual'] = df['income'] - df['prediction']

In [ ]:
# Covariance matrix for variance decomposition
Sigma = df[['income', 'education','prediction', 'residual']].cov()
Sigma

In [ ]:
# Variance components
total_variance =  Sigma.loc['income','income']  # TSS/n
prediction_variance = Sigma.loc['prediction','prediction']   # ESS/n  
residual_variance = Sigma.loc['residual','residual']      # RSS/n

In [ ]:
# Display TSS, ESS and RSS
print(f"Total variance: {total_variance:.2f}")
print(f"Explained variance: {prediction_variance:.2f}")
print(f"Residual variance: {residual_variance:.2f}")
print(f"Sum check: {prediction_variance + residual_variance:.2f}")


In [ ]:
# R² calculation
myRsquared = 1- residual_variance/total_variance
print(f"R-squared: {myRsquared:.2f}")


**Use statsmodels to estimate $\text{income} = \alpha + \beta \times \text{education}$. Comment regression statistics.**

In [ ]:
# OLS with statsmodels formula API
import statsmodels.formula.api as smf

# Define regression model with formula interface
model_1 = smf.ols(formula = 'income ~ education ', data = df)

# Fit the model to the data
res_1 = model_1.fit() 

print(res_1.summary())

**Use statsmodels to estimate $\text{income} = \alpha + \beta \times \text{prestige}$. Comment regression statistics.**

In [ ]:
# Regression with prestige as predictor
# Define regression model with formula interface
model_2 = smf.ols(formula = 'income ~ prestige', data = df)

# Fit the model to the data
res_2 = model_2.fit() 

print(res_2.summary())

**Use statsmodels to estimate $\text{income} = \alpha + \beta \times \text{education} + \beta_2 \times \text{prestige}$. Comment regression statistics.**

In [ ]:
# Multiple regression with education and prestige
# Define regression model with formula interface
model_3 = smf.ols(formula = 'income ~ education + prestige ', data = df)

# Fit the model to the data
res_3 = model_3.fit() 

print(res_3.summary())

**Which model would you recommend? For which purpose?**

**Plot the regression with prestige. Check visually normality of residuals**

In [ ]:
df['pred_prestige'] = res_2.predict(df['prestige'] )
df.head()

In [ ]:
plt.plot(df['prestige'], df['income'], 'o')  # Plot observed data points
plt.plot(df['prestige'], df['pred_prestige'], '-')  # Plot fitted regression line
plt.xlabel('Education')  # Label x-axis
plt.ylabel('Prestige')   # Label y-axis

In [ ]:
# Normality of residuals
x = df['prestige']
pred = df['pred_prestige']
actual =  df['income']
residuals = actual - pred

In [ ]:
plt.plot(x, residuals, 'o')

In [ ]:
plt.hist(residuals)

---

## Part 2: Taylor Rule

In 1993, John taylor, estimated, using US data the regression: $i_t = i^{\star} + \alpha_{\pi} \pi_t + \alpha_{\pi} y_t$ where $\pi_t$ is inflation and $y_t$ the output gap (let's say deviation from real gdp from the trend).
He found that both coefficients were not significantly different from $0.5$.
Our goal, is to replicate the same analysis.

__Import macro data from statsmodels (https://www.statsmodels.org/devel/datasets/generated/macrodata.html)__

In [ ]:
# Import statsmodels to access built-in datasets including macroeconomic data
import statsmodels

# Import statsmodels API for regression analysis
import statsmodels.api as sm

In [ ]:
# Load quarterly US macroeconomic data
ds = sm.datasets.macrodata.load_pandas()

In [ ]:
# Extract raw data from dataset object and preview
df = ds.raw_data
df.head()  # Display first 5 rows

**Create database with variables of interest including detrended GDP**

In [ ]:
# Extract key variables needed for Taylor Rule regression
gdp = df['realgdp']  # Real GDP
inflation = df['infl']  # Inflation rate
realint = df['realint']  # Real interest rate

In [ ]:
# Copy dataframe for analysis (will add computed variables like interest rate and output gap)
ddf = df

In [ ]:
# Preview the dataframe structure and contents
ddf.head()

Fisher relation: $r_t = i_t - \pi_t$

In [ ]:
# Nominal interest rate: i = r + π
ddf['ir'] = ddf['realint'] + ddf['infl']
ddf.head()

Detrend GDP using HP-filter (classical macroeconomic tool):

In [ ]:
# Import Hodrick-Prescott filter for trend-cycle decomposition
# HP filter is a classical tool in macroeconomics to extract business cycles
from statsmodels.tsa.filters.hp_filter import hpfilter

In [ ]:
# Apply HP filter to decompose GDP into trend and cyclical components
# cycle = deviations from trend (business cycle), trend = long-run path
cycle, trend = hpfilter(ddf['realgdp'])

In [ ]:
# Visualize trend-cycle decomposition of GDP
# Upper plot: trend line vs actual data; Lower plot: cyclical component alone
plt.subplot(211)
plt.plot(trend, label='Trend')
plt.plot(trend+cycle, label='Actual data')
plt.title("GDP Decomposition")
plt.legend(loc='upper left')

plt.subplot(212)
plt.plot(cycle)
plt.title("Cyclical Component")
plt.tight_layout()

In [ ]:
# Create output gap variable: (cycle / trend) * 100 as percentage
# Output gap measures deviation of actual GDP from potential GDP
ddf['gdp'] = cycle/trend*100

In [ ]:
# Preview updated dataframe with newly computed variables (ir and gdp)
ddf.head()

**Run the basic regression**

In [ ]:
# Import formula API for specifying regressions using R-style notation
from statsmodels.formula import api as sm

In [ ]:
# Basic Taylor Rule: interest rate responds to inflation and output gap
# Tests whether policy rate responds to inflation and economic slack
# "-1" suppresses the intercept (alternative specification without constant term)
model = sm.ols("ir ~ infl + gdp - 1", data=ddf)
results = model.fit()
print(results.summary())

**Which control variables would you add? Does it increase prediction power?**

In [ ]:
ddf.head()

In [ ]:
model = sm.ols("ir ~ infl + gdp + pop + unemp - 1", data=ddf)
results = model.fit()
print(results.summary())

**Comment on the regression results**